# ==========================================================================================
# METADATA-DRIVEN ETL PIPELINE: Federated Catalog (On-Prem DB) -> Temp View -> Bronze (CDC)
# ==========================================================================================

This notebook ingests raw data from on-prem databases (via Lakehouse Federation) into the Bronze layer.

**Control Tables:**
- `bronze.control.control_list_ingested_tables` — which tables to ingest
- `bronze.control.control_list_ingested_tables_constraints` — CDC tracking columns per table

**Pipeline Flow (per table):**
1. Read source data from federated catalog (`ims_live_catalog` / `dual_reporting_catalog`)
2. Compute `rowhash` (SHA-256) for change detection
3. For BUSINESS_KEY tables: dedup to latest row per CDC key
4. Stage data in a temporary view
5. MERGE into `bronze.ims.{table}` using CDC tracking columns
6. Log to `bronze.control.control_ingestion_log`

**source_status values:** New | Changed | Deleted

**Parameters:** Dropdown widgets allow filtering for ad-hoc runs.

In [0]:
%sql
/* Target catalog for bronze ingestion */
USE CATALOG bronze;

In [0]:
# ==========================================================================================
# Imports and Spark Configuration
# ==========================================================================================

from pyspark.sql.functions import (
    col, concat_ws, sha2, current_timestamp, lit, coalesce,
    row_number, from_utc_timestamp
)
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from datetime import datetime

# Enable Schema Evolution globally for Delta Merges
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

In [0]:
# ==========================================================================================
# Variables and Runtime Context
# ==========================================================================================

# Control table references
var_control_catalog = "bronze"
var_control_schema = "control"
var_control_table = f"{var_control_catalog}.{var_control_schema}.control_list_ingested_tables"
var_constraints_table = f"{var_control_catalog}.{var_control_schema}.control_list_ingested_tables_constraints"
var_log_table = f"{var_control_catalog}.{var_control_schema}.control_ingestion_log"

# Target
var_target_catalog = "bronze"

# ------------------------------------------------------------------------------------------
# EXTRACT RUNTIME CONTEXT
# ------------------------------------------------------------------------------------------
var_triggering_user_name = spark.sql("SELECT current_user()").collect()[0][0]

var_job_id_opt = dbutils.notebook.entry_point.getDbutils().notebook().getContext().jobId()
var_triggered_manual_or_automatic = "Automatic" if var_job_id_opt.isDefined() else "Manual"

In [0]:
# ==========================================================================================
# Dynamic Dropdown Parameters - populated from control table to eliminate typos
# ==========================================================================================
# Query control table for distinct values to populate dropdowns
dfr_ctrl = spark.table(var_control_table).filter("active_flag = true")

# Helper: extract sorted distinct non-empty values from a column, prepend blank option
def get_dropdown_choices(df, col_name):
    """Returns a list of distinct non-null, non-empty values with a blank option at the start."""
    rows = (
        df.select(col_name)
        .filter(f"{col_name} IS NOT NULL AND {col_name} != ''")
        .distinct()
        .orderBy(col_name)
        .collect()
    )
    values = [row[0] for row in rows]
    return [""] + values  # Blank = "all" / no filter

# Build choices for each parameter from the control table
lst_source_table_names = get_dropdown_choices(dfr_ctrl, "source_table_name")
lst_source_types = get_dropdown_choices(dfr_ctrl, "source_type")
lst_volume_names = get_dropdown_choices(dfr_ctrl, "unity_catalog_volume_name")
lst_directory_names = get_dropdown_choices(dfr_ctrl, "unity_catalog_directory_name")
lst_file_names = get_dropdown_choices(dfr_ctrl, "unity_catalog_file_name")
lst_source_catalogs = get_dropdown_choices(dfr_ctrl, "source_catalog")
lst_source_schemas = get_dropdown_choices(dfr_ctrl, "source_schema")

# Create dropdown widgets (leave blank to run for all tables in control list)
dbutils.widgets.dropdown("par_source_table_name", "", lst_source_table_names, "Source Table Name")
dbutils.widgets.dropdown("par_source_type", "", lst_source_types, "Source Type")
dbutils.widgets.dropdown("par_unity_catalog_volume_name", "", lst_volume_names, "Unity Catalog Volume Name")
dbutils.widgets.dropdown("par_unity_catalog_directory_name", "", lst_directory_names, "Unity Catalog Directory Name")
dbutils.widgets.dropdown("par_unity_catalog_file_name", "", lst_file_names, "Unity Catalog File Name")
dbutils.widgets.dropdown("par_source_catalog", "", lst_source_catalogs, "Source Catalog")
dbutils.widgets.dropdown("par_source_schema", "", lst_source_schemas, "Source Schema")

# Retrieve selected values
par_source_table_name = dbutils.widgets.get("par_source_table_name").strip()
par_source_type = dbutils.widgets.get("par_source_type").strip()
par_unity_catalog_volume_name = dbutils.widgets.get("par_unity_catalog_volume_name").strip()
par_unity_catalog_directory_name = dbutils.widgets.get("par_unity_catalog_directory_name").strip()
par_unity_catalog_file_name = dbutils.widgets.get("par_unity_catalog_file_name").strip()
par_source_catalog = dbutils.widgets.get("par_source_catalog").strip()
par_source_schema = dbutils.widgets.get("par_source_schema").strip()

In [0]:
# ==========================================================================================
# FETCH tables from CONTROL LIST + CDC tracking columns from CONSTRAINTS table
# ==========================================================================================

# Build base query with required filters
var_query = f"""
    SELECT * FROM {var_control_table}
    WHERE active_flag = true
      AND source_catalog IN ('ims_live_catalog', 'dual_reporting_catalog')
      AND target_schema = 'ims'
"""

# Apply parameter filters if provided
if par_source_table_name:
    var_query += f" AND source_table_name = '{par_source_table_name}'"
if par_source_type:
    var_query += f" AND source_type = '{par_source_type}'"
if par_source_catalog:
    var_query += f" AND source_catalog = '{par_source_catalog}'"
if par_source_schema:
    var_query += f" AND source_schema = '{par_source_schema}'"

var_query += " ORDER BY load_order"

dfr_control = spark.sql(var_query)
var_tables_to_process = dfr_control.collect()
var_total_tables = len(var_tables_to_process)

# Fetch CDC tracking columns (is_CDC_tracking_column = true) into a lookup dictionary
# Key: source_table_name -> Value: list of CDC column names (ordered by key_ordinal)
dfr_cdc = spark.sql(f"""
    SELECT source_table_name, constraint_column_name, constraint_type, key_ordinal
    FROM {var_constraints_table}
    WHERE is_CDC_tracking_column = true
    ORDER BY source_table_name, key_ordinal
""")

# Build dictionary: {table_name: [col1, col2, ...]}
var_cdc_columns_map = {}
for r in dfr_cdc.collect():
    tbl = r["source_table_name"]
    col_name = r["constraint_column_name"]
    if tbl not in var_cdc_columns_map:
        var_cdc_columns_map[tbl] = []
    var_cdc_columns_map[tbl].append(col_name)

# Also fetch identity PK columns for BUSINESS_KEY tables (needed for dedup ordering)
dfr_pk = spark.sql(f"""
    SELECT source_table_name, constraint_column_name
    FROM {var_constraints_table}
    WHERE constraint_type = 'PRIMARY_KEY_CONSTRAINT'
      AND is_CDC_tracking_column = false
""")
var_identity_pk_map = {r["source_table_name"]: r["constraint_column_name"] for r in dfr_pk.collect()}

print(f"Tables to process: {var_total_tables}")
print(f"Tables with CDC columns defined: {len(var_cdc_columns_map)}")

if not var_tables_to_process:
    print("No active tables found to process.")

Tables to process: 1
Tables with CDC columns defined: 95


In [0]:
# ==========================================================================================
# MAIN PIPELINE LOOP: Read Source -> Stage in Temp View -> MERGE into Bronze (CDC)
# ==========================================================================================

var_current_table = 0
var_failed_tables = []

print(f"\n--- Starting ingestion of {var_total_tables} tables ---")

for var_row in var_tables_to_process:
    var_src_table_name = var_row["source_table_name"]
    var_src_catalog = var_row["source_catalog"]
    var_src_schema = var_row["source_schema"]
    var_trg_schema = var_row["target_schema"]
    var_trg_table_name = var_row["target_table_name"]

    var_start_time = datetime.now()
    var_status = "SUCCESS"
    var_error_msg = "None"
    var_rows_processed = 0
    var_current_table += 1

    # Fully qualified target table name
    var_brz_table = f"{var_target_catalog}.{var_trg_schema}.{var_trg_table_name}"
    # Fully qualified source table name (federated catalog)
    var_source_fqn = f"{var_src_catalog}.{var_src_schema}.{var_src_table_name}"

    print(f"\n[{var_current_table}/{var_total_tables}] {var_src_table_name} -> {var_brz_table}")

    try:
        # ------------------------------------------------------------------------------
        # STEP 1: Get CDC tracking columns for this table
        # ------------------------------------------------------------------------------
        var_cdc_cols = var_cdc_columns_map.get(var_src_table_name, [])
        if not var_cdc_cols:
            # Fallback: use rowhash as merge key if no CDC columns defined
            print(f"  [WARN] No CDC columns for {var_src_table_name}. Using rowhash-only.")
            var_cdc_cols = ["rowhash"]
            var_use_rowhash_merge = True
        else:
            var_use_rowhash_merge = False

        # ------------------------------------------------------------------------------
        # STEP 2: Read source data from federated catalog
        # ------------------------------------------------------------------------------
        dfr_source = spark.table(var_source_fqn)

        # Lowercase all column names for consistency
        dfr_source = dfr_source.select([col(c).alias(c.lower()) for c in dfr_source.columns])

        # Exclude CDC columns and identity PK from rowhash (they are keys, not data)
        var_exclude_from_hash = set(c.lower() for c in var_cdc_cols)
        var_identity_pk = var_identity_pk_map.get(var_src_table_name)
        if var_identity_pk:
            var_exclude_from_hash.add(var_identity_pk.lower())
        var_hash_cols = sorted([c for c in dfr_source.columns if c not in var_exclude_from_hash])

        # Compute rowhash (SHA-256 of data columns only, NULL-safe)
        dfr_staged = dfr_source.withColumn(
            "rowhash",
            sha2(concat_ws("||" , *[coalesce(col(c).cast("string"), lit("__NULL__")) for c in var_hash_cols]), 256)
        )

        # For BUSINESS_KEY tables (log/changelog): dedup to latest row per CDC key
        var_identity_pk = var_identity_pk_map.get(var_src_table_name)
        if var_identity_pk and not var_use_rowhash_merge:
            identity_col = var_identity_pk.lower()
            if identity_col in [c.lower() for c in dfr_staged.columns]:
                cdc_cols_lower = [c.lower() for c in var_cdc_cols]
                w = Window.partitionBy(*cdc_cols_lower).orderBy(col(identity_col).desc())
                dfr_staged = dfr_staged.withColumn("_rn", row_number().over(w)) \
                    .filter("_rn = 1").drop("_rn")

        var_rows_processed = dfr_staged.count()

        # ------------------------------------------------------------------------------
        # STEP 3: Create temp view for staging
        # ------------------------------------------------------------------------------
        var_temp_view = f"vw_stg_{var_trg_table_name}"
        dfr_staged.createOrReplaceTempView(var_temp_view)

        # ------------------------------------------------------------------------------
        # STEP 4: Create bronze table if not exists, or MERGE (CDC)
        # ------------------------------------------------------------------------------
        var_cdc_cols_lower = [c.lower() for c in var_cdc_cols]

        if not spark.catalog.tableExists(var_brz_table):
            # First load: INSERT all rows as 'New'
            spark.sql(f"""
                CREATE TABLE {var_brz_table}
                USING DELTA
                TBLPROPERTIES (
                    'delta.autoOptimize.optimizeWrite' = 'true',
                    'delta.autoOptimize.autoCompact' = 'true',
                    'delta.columnMapping.mode' = 'name'
                )
                AS SELECT
                    *,
                    'New' AS source_status,
                    from_utc_timestamp(current_timestamp(), 'Australia/Melbourne') AS load_datetime
                FROM {var_temp_view}
            """)
            print(f"  [CREATED] {var_brz_table} with {var_rows_processed} rows (all 'New')")
        else:
            # Incremental CDC MERGE
            var_merge_condition = " AND ".join(
                [f"target.`{c}` = source.`{c}`" for c in var_cdc_cols_lower]
            )

            # Build column update/insert lists from staging view columns
            var_stg_columns = [c for c in dfr_staged.columns]
            var_insert_cols = var_stg_columns + ["rowhash", "source_status", "load_datetime"]
            var_insert_vals = [f"source.`{c}`" for c in var_stg_columns] + [
                "source.rowhash", "'New'",
                "from_utc_timestamp(current_timestamp(), 'Australia/Melbourne')"
            ]
            # For update (changed rows): update all business cols + rowhash + status + datetime
            var_update_set = ", ".join(
                [f"target.`{c}` = source.`{c}`" for c in var_stg_columns if c != "rowhash"]
                + [
                    "target.rowhash = source.rowhash",
                    "target.source_status = CASE WHEN target.source_status = 'Deleted' THEN 'New' ELSE 'Changed' END",
                    "target.load_datetime = from_utc_timestamp(current_timestamp(), 'Australia/Melbourne')"
                ]
            )

            var_merge_sql = f"""
                MERGE INTO {var_brz_table} target
                USING {var_temp_view} source
                ON {var_merge_condition}
                WHEN MATCHED AND (source.rowhash != target.rowhash OR target.source_status = 'Deleted') THEN
                    UPDATE SET {var_update_set}
                WHEN NOT MATCHED THEN
                    INSERT ({', '.join([f'`{c}`' for c in var_stg_columns] + ['source_status', 'load_datetime'])})
                    VALUES ({', '.join([f'source.`{c}`' for c in var_stg_columns] + ["'New'", "from_utc_timestamp(current_timestamp(), 'Australia/Melbourne')"])})
                WHEN NOT MATCHED BY SOURCE AND target.source_status != 'Deleted' THEN
                    UPDATE SET
                        target.source_status = 'Deleted',
                        target.load_datetime = from_utc_timestamp(current_timestamp(), 'Australia/Melbourne')
            """
            spark.sql(var_merge_sql)
            print(f"  [MERGED] {var_brz_table} — {var_rows_processed} source rows processed")

    except Exception as e:
        var_status = "FAILED"
        var_error_msg = str(e)[:500]
        var_failed_tables.append(var_src_table_name)
        print(f"  [ERROR] {var_src_table_name}: {var_error_msg}")

    finally:
        # Log ingestion result
        var_end_time = datetime.now()
        try:
            spark.sql(f"""
                INSERT INTO {var_log_table}
                (table_name, status, start_time, end_time, number_of_rows_processed,
                 triggered_manual_or_automatic, triggering_user_name, error_message)
                VALUES (
                    '{var_trg_table_name}', '{var_status}',
                    '{var_start_time.strftime("%Y-%m-%d %H:%M:%S")}',
                    '{var_end_time.strftime("%Y-%m-%d %H:%M:%S")}',
                    {var_rows_processed},
                    '{var_triggered_manual_or_automatic}',
                    '{var_triggering_user_name}',
                    '{var_error_msg.replace(chr(39), chr(39)+chr(39))}'
                )
            """)
        except Exception as log_err:
            print(f"  [LOG ERROR] Could not log: {log_err}")

# ==========================================================================================
# SUMMARY
# ==========================================================================================
print(f"\n{'='*60}")
print(f"INGESTION COMPLETE: {var_total_tables} tables processed")
print(f"  Successful: {var_total_tables - len(var_failed_tables)}")
print(f"  Failed: {len(var_failed_tables)}")
if var_failed_tables:
    print(f"  Failed tables: {var_failed_tables}")
print(f"{'='*60}")


--- Starting ingestion of 1 tables ---

[1/1] lstlines -> bronze.ims.lstlines
  [MERGED] bronze.ims.lstlines — 60 source rows processed

INGESTION COMPLETE: 1 tables processed
  Successful: 1
  Failed: 0


In [0]:
%sql
select * from anz_dev.bronze.ims_testtbl order by startdate_, TestTbl_id

TestTbl_id,TestTbl_name,TestTbl_age,TestTbl_city,rowhash,bronzepkhash,startdate_,enddate_,isactiveflag,isdeletedflag,islatestflag,MergeKey
1,Amir,50,Shiraz,ee2bdb3840f8cc508cfb825b7c178094631577b20eeae1410d9e931243ca4c31,9027e5c537e338a3ee150abdb09d70ae43330427f3a1d10ebac9c2fed740f2dc,2026-05-19T04:27:20.646718Z,2026-05-19T04:37:09.110675Z,false,true,true,null
2,Kamran,48,Kerachi,6b89eb839c5ec685a7f0b71d18bb97dbfce26eee06b34022012b2c0a6a2890b5,57f83cdf224aede08d5e7bf3ea8806fb06e8275a316570bcc190e7d2d9c624a4,2026-05-19T04:27:20.646718Z,2026-05-19T04:37:09.110675Z,false,true,true,null
3,Matt,60,Sydney,794cb7a77a14862ddb5d25a9bb71ef96ec45be7bcf5c08485b2597890f71d177,148fb969182e7568114d74dfa7115b8d89662d83a2bb89ba8981bfd430519d79,2026-05-19T04:27:20.646718Z,2026-05-20T05:36:44.546946Z,false,true,true,null
2,Kamran,48,Multan,1f2e1c726827e0f4e5ecfb1de09e6e641a436d2dadb3949bbff53450d1f034cb,7f0188b8d91a63024f2475dbe399994ba9b16cc8f99818baae3ba38bb255464f,2026-05-19T04:37:12.340133Z,1900-01-01T00:00:00Z,true,false,true,null
4,Hemi,45,Melbourne,e7541c2d6161c7f496668f554f1a3f1a5aeb343c2e74a8c8679a9d05eb1781af,1d228396b54a3d111919a5ce651f59e6efbdc828a0cc19b50c4761f34fd3a5a9,2026-05-19T04:37:12.340133Z,2026-05-20T05:36:44.546946Z,false,true,true,null
4,Hemi,45,Adelaide,161431f0e37690c3244e9911d1203f9dd4d89f687ec42a7debe7818c8830cc02,d4f872f47ec475fe5da9d5bfcf3ad6cfb89d2baf1549e2ddace310e0395a701f,2026-05-20T05:36:50.610085Z,1900-01-01T00:00:00Z,true,false,true,null
5,Meena,18,Deh,73c430e3e215b4f988513db915367fec6e8c690916380e6c03462b08be129770,7effeb2b51c16d76c18babcc62eae5a386d843727b711c8e34aead109767ffac,2026-05-20T05:36:50.610085Z,1900-01-01T00:00:00Z,true,false,true,null


In [0]:
%sql
select * from anz_dev.default.testtbl order by id

Id,Name,Age,City
2,Kamran,49,Multan
4,Hemi,45,Adelaide
5,Meena,18,Deh
6,Achi,35,Melbourne


In [0]:
# ==========================================================================================
# AD HOC: DROP EXISTING STAGING AND BRONZE TABLES + CLEAN UP AZURE STORAGE
# ==========================================================================================
# Run this cell manually/ad hoc when you need to drop and recreate all staging and bronze
# tables. This is NOT part of the regular pipeline execution.
# Tables excluded from dropping: ims_custom_displaystatus, ims_custom_eventtypes
# Also removes underlying Delta files from Azure storage to avoid orphaned data.
# ==========================================================================================
"""

var_excluded_tables = ['ims_quotedisplaystatus', 'ims_transactioneventtypes']

# Get target catalog from first table in the control list (all tables use the same target catalog)
var_drop_catalog = var_tables_to_process[0]['target_catalog'] if var_tables_to_process else spark.catalog.currentCatalog()

print("--- Dropping existing staging and bronze tables (except excluded tables) ---")
print("--- Also cleaning up underlying Azure storage files ---\n")

# Get list of all tables in bronze schema
var_brz_tables = spark.sql(f"SHOW TABLES IN {var_drop_catalog}.{var_bronze_schema_name}").collect()

var_drop_count = 0
var_storage_cleaned_count = 0

for var_brz_table_row in var_brz_tables:
    var_existing_table_name = var_brz_table_row['tableName'].lower()
    var_full_table_name = f"{var_drop_catalog}.{var_bronze_schema_name}.{var_existing_table_name}"
    
    # Check if it's a staging table (starts with stg_) or a bronze IMS table (starts with ims_)
    #if var_existing_table_name.startswith('stg_') or (var_existing_table_name.startswith('ims_') and var_existing_table_name not in var_excluded_tables):
    if var_existing_table_name.startswith('ims_dual_figtree_claims') or (var_existing_table_name.startswith('stg_ims_dual_figtree_claims') and var_existing_table_name not in var_excluded_tables):
        # Skip control tables
        if var_existing_table_name.startswith('control_'):
            continue
        try:
            # Get storage location before dropping (for external tables with explicit paths)
            var_storage_location = None
            try:
                var_storage_location = spark.sql(f"DESCRIBE DETAIL {var_full_table_name}").select("location").collect()[0][0]
            except Exception:
                pass  # Table might be managed or metadata might be stale
            
            # Drop the table metadata
            spark.sql(f"DROP TABLE IF EXISTS {var_full_table_name}")
            var_drop_count += 1
            print(f"  Dropped: {var_full_table_name}")
            
            # Clean up underlying Azure storage files if location was found
            if var_storage_location:
                try:
                    dbutils.fs.rm(var_storage_location, True)
                    var_storage_cleaned_count += 1
                    print(f"    -> Cleaned storage: {var_storage_location}")
                except Exception as e:
                    print(f"    -> Warning: Could not clean storage at {var_storage_location}: {str(e)[:100]}")
        except Exception as e:
            print(f"  Warning: Could not drop {var_full_table_name}: {str(e)[:100]}")

print(f"\n--- Finished dropping {var_drop_count} table(s). Cleaned {var_storage_cleaned_count} storage location(s). ---")
print(f"--- Excluded: {', '.join(var_excluded_tables)} ---")
"""

'\n\nvar_excluded_tables = [\'ims_quotedisplaystatus\', \'ims_transactioneventtypes\']\n\n# Get target catalog from first table in the control list (all tables use the same target catalog)\nvar_drop_catalog = var_tables_to_process[0][\'target_catalog\'] if var_tables_to_process else spark.catalog.currentCatalog()\n\nprint("--- Dropping existing staging and bronze tables (except excluded tables) ---")\nprint("--- Also cleaning up underlying Azure storage files ---\n")\n\n# Get list of all tables in bronze schema\nvar_brz_tables = spark.sql(f"SHOW TABLES IN {var_drop_catalog}.{var_bronze_schema_name}").collect()\n\nvar_drop_count = 0\nvar_storage_cleaned_count = 0\n\nfor var_brz_table_row in var_brz_tables:\n    var_existing_table_name = var_brz_table_row[\'tableName\'].lower()\n    var_full_table_name = f"{var_drop_catalog}.{var_bronze_schema_name}.{var_existing_table_name}"\n    \n    # Check if it\'s a staging table (starts with stg_) or a bronze IMS table (starts with ims_)\n    #if 

In [0]:
# Though this table has been created and maintained separately, yet its creation script written here for explanation purposes. From the below table we will get all the tables to be ingested into the Bronze Layer
spark.sql(f"""
	CREATE TABLE IF NOT EXISTS bronze.control_list_ingested_tables(
    sk_control_list_ingested_tables BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    source_table_name STRING,
    source_catalog STRING,
    source_schema STRING,
    target_catalog STRING,
    target_schema STRING,
    target_table_name STRING,
    target_table_columns_prefix STRING,
    primary_key STRING,     -- e.g. HashRow
    active_flag BOOLEAN DEFAULT true,     -- Y/N
    load_order BIGINT GENERATED BY DEFAULT AS IDENTITY,
    DMLDateTime TIMESTAMP DEFAULT from_utc_timestamp(CURRENT_TIMESTAMP(), 'Australia/Melbourne'),
  	CONSTRAINT pk_control_list_ingested_tables PRIMARY KEY (table_name, source_catalog, source_schema)
    )
    USING DELTA
    LOCATION 'abfss://dev@anzdevstorage.dfs.core.windows.net/brz/ims/control_list_ingested_tables'
    TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.minReaderVersion' = '3',	--This property allows renaming or Datatype alteration of a column later on if needed
    'delta.minWriterVersion' = '7',	--This property allows renaming or Datatype alteration of a column later on if needed 
    'delta.columnMapping.mode' = 'name',	--This property allows renaming or Datatype alteration of a column later on if needed
    'delta.feature.allowColumnDefaults' = 'enabled'
    )
""")

DataFrame[]

In [0]:
#Logging Table (for observability)
# Though this table can also be created separately if doesn't exist already, and reused to log the ingestion process for other than claims tables as well

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.control_ingestion_log(
    sk_control_ingestion_log BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    table_name STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    number_of_rows_processed BIGINT,
    triggered_manual_or_automatic STRING,
    triggering_user_name STRING,
    error_message STRING,
    DMLDateTime TIMESTAMP DEFAULT from_utc_timestamp(current_timestamp(), 'Australia/Melbourne'),
  	CONSTRAINT pk_control_ingestion_log PRIMARY KEY (table_name, start_time)
)
USING DELTA
LOCATION 'abfss://dev@anzdevstorage.dfs.core.windows.net/brz/ims/control_ingestion_log'
TBLPROPERTIES (
'delta.autoOptimize.optimizeWrite' = 'true',
'delta.autoOptimize.autoCompact' = 'true',
'delta.minReaderVersion' = '3',	--This property allows renaming or Datatype alteration of a column later on if needed
'delta.minWriterVersion' = '7',	--This property allows renaming or Datatype alteration of a column later on if needed 
'delta.columnMapping.mode' = 'name',	--This property allows renaming or Datatype alteration of a column later on if needed
'delta.feature.allowColumnDefaults' = 'enabled'
    )
""")

DataFrame[]

In [0]:
#Below code renames columns and converts data types in the bronze table to match new naming convention.
#Though not needed to run each time, leaving it here as the system is in development phase.
"""
# ==========================================================================================
# RENAME columns and convert data types in bronze table
# ==========================================================================================
var_brz_rename_table = "anz_dev.bronze.tblClaims_PolicyInformation"

if spark.catalog.tableExists(var_brz_rename_table):
    # Get current columns to check which renames are needed
    brz_columns = [c.name for c in spark.table(var_brz_rename_table).schema]
    
    # --- Step 1: Rename columns to match new convention ---
    rename_map = {
        "EffectiveFrom": "StartDate_",
        "EffectiveTo": "EndDate_",
        "ActiveRecord": "IsActiveFlag",
        "DeletedRecord": "IsDeletedFlag",
        "LatestRecord": "IsLatestFlag"
    }
    
    for old_name, new_name in rename_map.items():
        if old_name in brz_columns and new_name not in brz_columns:
            spark.sql(f"ALTER TABLE {var_brz_rename_table} RENAME COLUMN `{old_name}` TO `{new_name}`")
            print(f"  Renamed: {old_name} -> {new_name}")
        elif new_name in brz_columns:
            print(f"  Already exists: {new_name} (skip)")
    
    # Handle rowhash -> RowHash
    # Note: Delta with column mapping treats rowhash/RowHash as the same column (case-insensitive),
    # so no rename is needed — SQL references to RowHash will resolve to rowhash automatically.
    brz_columns = [c.name for c in spark.table(var_brz_rename_table).schema]
    if "rowhash" in brz_columns or "RowHash" in brz_columns:
        print(f"  RowHash column OK (Delta resolves case-insensitively)")
    
    # Add BronzePKHash column if missing
    brz_columns = [c.name for c in spark.table(var_brz_rename_table).schema]
    if "BronzePKHash" not in brz_columns:
        spark.sql(f"ALTER TABLE {var_brz_rename_table} ADD COLUMN BronzePKHash STRING")
        print(f"  Added column: BronzePKHash")
    else:
        print(f"  BronzePKHash already exists (skip)")
    
    # --- Step 2: Convert INT flag columns to BOOLEAN ---
    brz_schema = {f.name: str(f.dataType) for f in spark.table(var_brz_rename_table).schema}
    
    flag_columns = ["IsActiveFlag", "IsDeletedFlag", "IsLatestFlag"]
    for flag_col in flag_columns:
        col_type = brz_schema.get(flag_col, "")
        if "Integer" in col_type or "Long" in col_type:
            temp_col = f"_{flag_col}_bool"
            # Add new BOOLEAN column, populate from old INT column, drop old, rename new
            spark.sql(f"ALTER TABLE {var_brz_rename_table} ADD COLUMN `{temp_col}` BOOLEAN")
            spark.sql(f"UPDATE {var_brz_rename_table} SET `{temp_col}` = CAST(`{flag_col}` AS BOOLEAN)")
            spark.sql(f"ALTER TABLE {var_brz_rename_table} DROP COLUMN `{flag_col}`")
            spark.sql(f"ALTER TABLE {var_brz_rename_table} RENAME COLUMN `{temp_col}` TO `{flag_col}`")
            print(f"  Converted {flag_col}: INT -> BOOLEAN")
        elif flag_col in brz_schema:
            print(f"  {flag_col} already {col_type} (skip)")
        else:
            print(f"  {flag_col} not found in table (skip)")
    
    print(f"\nColumn renames and type conversions complete for {var_brz_rename_table}")
else:
    print(f"Table {var_brz_rename_table} does not exist yet - will be created by the pipeline.")
"""

'\n# ==========================================================================================\n# RENAME columns and convert data types in bronze table\n# ==========================================================================================\nvar_brz_rename_table = "anz_dev.brz.tblClaims_PolicyInformation"\n\nif spark.catalog.tableExists(var_brz_rename_table):\n    # Get current columns to check which renames are needed\n    brz_columns = [c.name for c in spark.table(var_brz_rename_table).schema]\n    \n    # --- Step 1: Rename columns to match new convention ---\n    rename_map = {\n        "EffectiveFrom": "StartDate_",\n        "EffectiveTo": "EndDate_",\n        "ActiveRecord": "IsActiveFlag",\n        "DeletedRecord": "IsDeletedFlag",\n        "LatestRecord": "IsLatestFlag"\n    }\n    \n    for old_name, new_name in rename_map.items():\n        if old_name in brz_columns and new_name not in brz_columns:\n            spark.sql(f"ALTER TABLE {var_brz_rename_table} RENAME COLUM

In [0]:
# ==========================================================================================
# AD HOC: Add source_table_has_primarykey (BOOLEAN) column and populate with values
# ==========================================================================================

# Step 1: Add the new column (idempotent — skips if column already exists)
existing_cols = [f.name.lower() for f in spark.table("anz_dev.bronze.control_list_ingested_tables").schema]
if "source_table_has_primarykey" not in existing_cols:
    spark.sql("ALTER TABLE anz_dev.bronze.control_list_ingested_tables ADD COLUMNS (source_table_has_primarykey BOOLEAN)")
    print("Column source_table_has_primarykey added.")
else:
    print("Column source_table_has_primarykey already exists — skipping ALTER TABLE.")

# Step 2: Merge the provided mapping values into the new column
spark.sql("""
    MERGE INTO anz_dev.bronze.control_list_ingested_tables AS target
    USING (
      SELECT 'CLI_CLIENT'                                    AS source_table_name, FALSE AS source_table_has_primarykey UNION ALL
      SELECT 'DUAL_ClaimantCustomData_AffectedCoverage',                           TRUE  UNION ALL
      SELECT 'DUAL_ClaimantCustomData_WatchList',                                  TRUE  UNION ALL
      SELECT 'DUAL_CustomClaimantData',                                            TRUE  UNION ALL
      SELECT 'DUAL_CustomClaimData',                                               TRUE  UNION ALL
      SELECT 'DUAL_CustomTPAClaimsInfo',                                           TRUE  UNION ALL
      SELECT 'Dual_FigTreeClaims',                                                 FALSE UNION ALL
      SELECT 'Dual_FigTreeClaims_AU',                                              FALSE UNION ALL
      SELECT 'Dual_FigTreeClaims_ForClaimImport_AU',                               FALSE UNION ALL
      SELECT 'DUAL_lstClaimsCoverageTypes',                                        TRUE  UNION ALL
      SELECT 'DUAL_lstClaimsDeclinationReasons',                                   TRUE  UNION ALL
      SELECT 'DUAL_lstClaimsRecoveryTypes',                                        TRUE  UNION ALL
      SELECT 'DUAL_lstEstReserveQuarter',                                          TRUE  UNION ALL
      SELECT 'DUAL_lstLitigationStatus',                                           TRUE  UNION ALL
      SELECT 'DUAL_lstLossNature',                                                 TRUE  UNION ALL
      SELECT 'DUAL_lstQuarterAdded',                                               TRUE  UNION ALL
      SELECT 'DUAL_lstSeverityOfLoss',                                             TRUE  UNION ALL
      SELECT 'DUAL_VendorEntity',                                                  TRUE  UNION ALL
      SELECT 'DUALAPAC_ClaimsPanel',                                               FALSE UNION ALL
      SELECT 'DUALAPAC_lstClaims_APRACodes',                                       TRUE  UNION ALL
      SELECT 'IAC_INSURERSONPOLICY',                                               FALSE UNION ALL
      SELECT 'ibo_insurerbranch',                                                  FALSE UNION ALL
      SELECT 'ICA_INSURER_ACCOUNTING',                                             FALSE UNION ALL
      SELECT 'lstClaims_CatastropheCodes',                                         TRUE  UNION ALL
      SELECT 'lstClaims_ClaimActivities',                                          FALSE UNION ALL
      SELECT 'lstClaims_ClaimStatus',                                              TRUE  UNION ALL
      SELECT 'lstClaims_CoverageTypes',                                            TRUE  UNION ALL
      SELECT 'lstClaims_OutsideAdjusters',                                         TRUE  UNION ALL
      SELECT 'lstClaims_ReservePaymentSubTypes',                                   TRUE  UNION ALL
      SELECT 'lstClaims_ReservePaymentTypes',                                      TRUE  UNION ALL
      SELECT 'lstCurrencies',                                                      TRUE  UNION ALL
      SELECT 'PLY_Policy',                                                         FALSE UNION ALL
      SELECT 'tblClaims_ActivityLog',                                              TRUE  UNION ALL
      SELECT 'tblClaims_Claim',                                                    TRUE  UNION ALL
      SELECT 'tblClaims_ClaimAccidentInformation',                                 TRUE  UNION ALL
      SELECT 'tblClaims_Claimants',                                                TRUE  UNION ALL
      SELECT 'tblClaims_ClaimPayees',                                              TRUE  UNION ALL
      SELECT 'tblClaims_PolicyInformation',                                        TRUE  UNION ALL
      SELECT 'tblClaims_ReservePayments',                                          TRUE  UNION ALL
      SELECT 'Test_ml',                                                            FALSE UNION ALL
      SELECT 'TestTbl',                                                            FALSE
    ) AS source
    ON target.source_table_name = source.source_table_name
    WHEN MATCHED THEN
      UPDATE SET target.source_table_has_primarykey = source.source_table_has_primarykey
""")
print("MERGE completed: source_table_has_primarykey values updated.")

# Step 3: Verify results
display(spark.sql("""
    SELECT source_table_name, source_table_has_primarykey
    FROM   anz_dev.bronze.control_list_ingested_tables
    ORDER BY source_table_name
"""))

Column source_table_has_primarykey added.
MERGE completed: source_table_has_primarykey values updated.


source_table_name,source_table_has_primarykey
CLI_CLIENT,false
DUALAPAC_ClaimsPanel,false
DUALAPAC_lstClaims_APRACodes,true
DUAL_ClaimantCustomData_AffectedCoverage,true
DUAL_ClaimantCustomData_WatchList,true
DUAL_CustomClaimData,true
DUAL_CustomClaimantData,true
DUAL_CustomTPAClaimsInfo,true
DUAL_VendorEntity,true
DUAL_lstClaimsCoverageTypes,true


In [0]:
# Though this table has been created and maintained separately, yet its creation script written here for explanation purposes. From the below table we will get the Primary or Unique Key contraints info of the tables to be ingested. This information will help us to identify New, Modified or Deleted source rows
spark.sql(f"""
CREATE TABLE bronze.control_list_ingested_tables_constraints
(
    sk_control_list_ingested_tables_constraints BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),

    -- FK to control table
    sk_control_list_ingested_tables BIGINT NOT NULL,

    -- Denormalized for easier troubleshooting
    source_table_name STRING NOT NULL,

    -- Constraint metadata
    constraint_name STRING NOT NULL,
    constraint_type STRING NOT NULL,

    -- Column participating in constraint
    constraint_column_name STRING NOT NULL,
    key_ordinal INT NOT NULL,

    -- SCD2 Business Key Indicator
    is_business_key BOOLEAN DEFAULT FALSE NOT NULL,
    is_scd2_tracked_column BOOLEAN DEFAULT FALSE,

    -- Audit columns
    dmldatetime TIMESTAMP DEFAULT from_utc_timestamp(CURRENT_TIMESTAMP(), 'Australia/Melbourne'),

    CONSTRAINT pk_control_list_ingested_tables_constraints
    PRIMARY KEY (
        sk_control_list_ingested_tables_constraints
    )
    )
        USING DELTA
        LOCATION 'abfss://dev@anzdevstorage.dfs.core.windows.net/brz/ims/control_list_ingested_tables_constraints'
        TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true',
        'delta.minReaderVersion' = '3',	--This property allows renaming or Datatype alteration of a column later on if needed
        'delta.minWriterVersion' = '7',	--This property allows renaming or Datatype alteration of a column later on if needed 
        'delta.columnMapping.mode' = 'name',	--This property allows renaming or Datatype alteration of a column later on if needed
        'delta.feature.allowColumnDefaults' = 'enabled'
        )
""");